# Task 2 — Exploratory Data Analysis
Ethiopia Financial Inclusion Forecasting

In [1]:
import sys
sys.path.insert(0, "../src")
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
from data_loader import load_unified_data, load_reference_codes, validate_against_reference, split_by_record_type, get_indicator_series
df = load_unified_data()
ref = load_reference_codes()
df.head()

,record_id,record_type,category,pillar,indicator,indicator_code,indicator_direction,value_numeric,value_text,value_type,unit,observation_date,period_start,period_end,fiscal_year,gender,location,region,source_name,source_type,source_url,confidence,related_indicator,parent_id,relationship_type,impact_direction,impact_magnitude,magnitude_estimate_pct,lag_months,evidence_basis,original_text,collected_by,collection_date,notes
0,REC_0001,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,22.0,NaN,percentage,%,2014-12-31,NaT,NaT,2014,all,national,NaN,Global Findex 2014,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN
1,REC_0002,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,35.0,NaN,percentage,%,2017-12-31,NaT,NaT,2017,all,national,NaN,Global Findex 2017,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN
2,REC_0003,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,46.0,NaN,percentage,%,2021-12-31,NaT,NaT,2021,all,national,NaN,Global Findex 2021,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN
3,REC_0004,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,56.0,NaN,percentage,%,2021-12-31,NaT,NaT,2021,male,national,NaN,Global Findex 2021,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN
4,REC_0005,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,36.0,NaN,percentage,%,2021-12-31,NaT,NaT,2021,female,national,NaN,Global Findex 2021,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN


In [2]:
import matplotlib.pyplot as plt
import sys; sys.path.insert(0, "../src")
from eda import (record_type_summary, confidence_summary, temporal_coverage,
                  plot_access_trajectory, plot_event_overlay, gender_gap_summary, key_insights)

## Dataset overview by record type and pillar

In [3]:
record_type_summary(df)

,record_type,pillar,count
0,event,NaN,10
1,impact_link,ACCESS,7
2,impact_link,USAGE,9
3,observation,ACCESS,14
4,observation,AFFORDABILITY,1
5,observation,GENDER,4
6,observation,USAGE,12
7,target,ACCESS,2
8,target,GENDER,1


## Data quality: confidence distribution

In [4]:
confidence_summary(df)

,record_type,confidence,count
0,impact_link,estimated,6
1,impact_link,low,10
2,observation,high,28
3,observation,medium,3


## Temporal coverage — which years have data for which indicators

In [5]:
temporal_coverage(df)

year,2014,2017,2021,2023,2024,2025
indicator_code,,,,,,
ACC_4G_COV,0,0,0,1,0,1
ACC_FAYDA,0,0,0,0,1,2
ACC_MM_ACCOUNT,0,0,1,0,1,0
ACC_MOBILE_PEN,0,0,0,0,0,1
ACC_OWNERSHIP,1,1,3,0,1,0
AFF_DATA_INCOME,0,0,0,0,1,0
GEN_GAP_ACC,0,0,1,0,1,0
GEN_GAP_MOBILE,0,0,0,0,1,0
GEN_MM_SHARE,0,0,0,0,1,0


## Access: Account Ownership trajectory, 2011–2024
Growth decelerated sharply in the most recent round: +13pp (2014-17), +11pp (2017-21), only **+3pp** (2021-24).

In [6]:
fig = plot_access_trajectory(df)
fig

<Figure size 800x500 with 1 Axes>

In [7]:
acc = get_indicator_series(df, "ACC_OWNERSHIP")
acc["growth_pp"] = acc["value_numeric"].diff()
acc

,observation_date,value_numeric,confidence,source_name,growth_pp
0,2014-12-31,22.0,high,Global Findex 2014,NaN
1,2017-12-31,35.0,high,Global Findex 2017,13.0
2,2021-12-31,46.0,high,Global Findex 2021,11.0
3,2024-11-29,49.0,high,Global Findex 2024,3.0


## Gender gap in account ownership

In [8]:
gender_gap_summary(df)

,observation_date,gender,value_numeric
3,2021-12-31,male,56.0
4,2021-12-31,female,36.0


## Investigating the 2021–2024 slowdown
Mobile money accounts more than doubled over the same window — so why did headline Access barely move?

In [9]:
mm = get_indicator_series(df, "ACC_MM_ACCOUNT")
mm

,observation_date,value_numeric,confidence,source_name
0,2021-12-31,4.70,high,Global Findex 2021
1,2024-11-29,9.45,high,Global Findex 2024


Ethiopia's market nuance (from the enrichment guide): mobile-money-only users are rare (~0.5%). Most new mobile money registrants already had a bank account, so registration growth does not convert 1:1 into new Findex 'account owners'. This is modeled explicitly in the impact_link table (`IMP_0003`/`IMP_0004`, `relationship_type: indirect`).

## Event timeline overlaid on indicator trends

In [10]:
fig = plot_event_overlay(df, "ACC_MM_ACCOUNT")
fig

<Figure size 1000x500 with 1 Axes>

In [11]:
fig = plot_event_overlay(df, "USG_P2P_COUNT")
fig

<Figure size 1000x500 with 1 Axes>

## P2P vs ATM: the crossover

In [12]:
p2p = get_indicator_series(df, "USG_P2P_COUNT")
atm = get_indicator_series(df, "USG_ATM_COUNT")
print("P2P:\n", p2p, "\n\nATM:\n", atm)

P2P:
   observation_date  value_numeric confidence              source_name
0       2024-07-07     49700000.0       high  EthSwitch Annual Report
1       2025-07-07    128300000.0       high  EthSwitch Annual Report 

ATM:
   observation_date  value_numeric confidence              source_name
0       2025-07-07    119300000.0       high  EthSwitch Annual Report


## Five key insights

In [13]:
for i, ins in enumerate(key_insights(df), 1):
    print(f"{i}. {ins}\n")

1. Account ownership growth decelerated sharply in the most recent survey round: +13pp (2014-17), +11pp (2017-21), but only +3pp (2021-24), despite mobile money accounts more than doubling (4.7% -> 9.4%) over the same window.

2. The registration-to-Access gap is explained by Ethiopia's market structure: mobile-money-only users are rare (~0.5%), so most new mobile money registrants already had a bank account -- registrations do not convert 1:1 into new account owners in the Findex sense.

3. The gender gap in account ownership narrowed only marginally (20pp in 2021 to 18pp in 2024), suggesting mobile money expansion has not yet meaningfully closed Ethiopia's financial inclusion gender divide.

4. P2P digital transfers surpassed ATM cash withdrawals in volume for the first time in FY2024/25 (128.3M P2P transactions vs. 119.3M ATM transactions), a structural shift toward digital-first payment behavior even though survey-measured Access growth has stalled.

5. Nearly all of the largest-ma